# Pipeline complet NdapuSpeech — exécution bout en bout

Ce notebook exécute **toutes les étapes** du pipeline ASR Wolof dans l'ordre :

| # | Étape | Durée estimée |
|---|-------|---------------|
| 1 | Téléchargement des datasets existants | ~10 min |
| 2 | Collecte YouTube (optionnel) | ~30 min |
| 3 | Construction du dataset unifié | ~5 min |
| 4b | Pré-calcul Mel + labels (Arrow, 0x I/O) | ~5 min (une fois) |
| 4 | Entraînement tokenizer BPE Wolof | ~2 min |
| 5 | Fine-tuning Whisper optimisé (SDPA + LoRA 4-bit, group_by_length) | ~minutes/époque (RTX 5060 8Go) |
| 6 | Entraînement KenLM n-gram | ~5 min |
| 7 | Évaluation WER/CER vs M-Kiriku | ~5 min |
| 8 | Inférence sur des fichiers audio | ~1 min |

> **Prérequis** : Python ≥ 3.11, GPU CUDA disponible (RTX recommandé),
> `ffmpeg` et `yt-dlp` installés dans le PATH.

## Prérequis — installation des packages

In [1]:
%pip install -e "..[audio]"

Obtaining file:///C:/Users/ROG%20ZEPHYRUS%20G14/OneDrive%20-%20Universit%C3%A9%20Cheikh%20Anta%20DIOP%20de%20DAKAR/Desktop/B2B/NdapuSpeech
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for ndapuspeech (pyproject.toml): started
  Building editable for ndapuspeech (pyproject.toml): finished with status 'done'
  Created wheel for ndapuspeech: filename=ndapuspeech-0.1.0-0.editable-py3-none-any.whl size=3584 sha256=323a3b41398447ef47e0874a72971c1a8f1062496a315881b92b1cc469739b55
  Stored in directory: C:\Users\ROG ZEPHYRUS G14\Ap

In [2]:
import logging, time, torch

logging.basicConfig(level=logging.INFO, format='%(message)s')
logging.getLogger().setLevel(logging.INFO)

from ndapuspeech import config, data, models, evaluation, inference, prepare
from ndapuspeech.training import TrainingConfig, train_model
from ndapuspeech.kenlm import train_kenlm

config.ensure_dirs()

print(f"CUDA : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Base model : {config.BASE_MODEL}")
print(f"Language : {config.LANGUAGE}")
print(f"Config chargée : {len(config.get_config())} sections")

CUDA : True — NVIDIA GeForce RTX 5060 Laptop GPU
Base model : openai/whisper-large-v3-turbo
Language : wo
Config chargée : 3 sections


---
## Étape 1 — Téléchargement des datasets

In [3]:
t0 = time.time()
results = data.download_all()
print(f"\n--- Téléchargements ({time.time()-t0:.0f}s) ---")
for source, path in results.items():
    status = 'OK' if path else 'SKIPPED'
    print(f"  {source:16s} {status}")

=== Kallaama (OpenSLR 151) — 55h agriculture ===
[SKIP] Kallaama already exists at C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh Anta DIOP de DAKAR\Desktop\B2B\NdapuSpeech\data\raw\kallaama
=== ALFFA (OpenSLR 25) — 18h read speech ===
[SKIP] ALFFA already exists at C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh Anta DIOP de DAKAR\Desktop\B2B\NdapuSpeech\data\raw\alffa
=== Google FLEURS Wolof — ~10h multi-domain ===
c:\Users\ROG ZEPHYRUS G14\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[SKIP] fleurs already exists at C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh Anta DIOP de DAKAR\Desktop\B2B\NdapuSpeech\data\raw\fleurs
=== WolBanking77 — 4h banking ===
[SKIP] wolbanking77 already exists at C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh An


--- Téléchargements (8s) ---
  kallaama         OK
  alffa            OK
  fleurs           OK
  wolbanking77     OK


---
## Étape 3 — Construction du dataset unifié train / val / test

In [4]:
t0 = time.time()
summary = data.build_unified_dataset(
    output_dir=config.PROCESSED_DATA_DIR / 'ndapuspeech_ds',
    train_ratio=config.TRAIN_SPLIT,
)
print(f"\nDataset ({time.time()-t0:.0f}s) :")
print(f"  Segments  : {summary['n_samples']}")
print(f"  Durée tot : {summary['total_hours']:.1f} h")
for src, hrs in sorted(summary['by_source'].items(), key=lambda x: -x[1]):
    print(f"  {src:20s} {hrs:6.1f} h")
print(f"  Splits    : {summary['splits']}")

[SKIP] Unified dataset already exists at C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh Anta DIOP de DAKAR\Desktop\B2B\NdapuSpeech\data\processed\ndapuspeech_ds



Dataset (0s) :
  Segments  : 48903
  Durée tot : 80.9 h
  kallaama               48.3 h
  alffa                  21.4 h
  fleurs                 11.2 h
  Splits    : {'train': 41567, 'val': 4888, 'test': 2448}


---
## Étape 4 — Entraînement du tokenizer BPE Wolof

In [5]:
t0 = time.time()
tok_result = models.train_tokenizer_pipeline(
    vocab_size=config.DATA['tokenizer']['vocab_size'],
    extend_whisper=False,
)
print(f"\nTokenizer ({time.time()-t0:.0f}s) :")
print(f"  Pièces BPE   : {tok_result['n_pieces']}")
print(f"  Corpus chars  : {tok_result['corpus_chars']}")
print(f"  Étendu vers   : {tok_result['extended_tokenizer']}")


Tokenizer (1s) :
  Pièces BPE   : 9997
  Corpus chars  : 5201475
  Étendu vers   : None


---
## Étape 4b — Pré-calcul Mel + labels 

APRÈS le tokenizer (les labels en dépendent). Une seule fois : `datasets.map(num_proc)` → `save_to_disk`. L'entraînement ne fait ensuite qu'une copie RAM → VRAM.

In [6]:
t0 = time.time()
pre_dir = prepare.prepare_feature_dataset(
    data_dir=config.PROCESSED_DATA_DIR / 'ndapuspeech_ds',
    output_dir=config.TRAINING_PREPROCESSED_DIR,
    processor_id=config.BASE_MODEL,
    tokenizer_dir=None,
    num_proc=config.TRAINING_NUM_PROC,
    force=True,
)
print(f"\nPré-calcul ({time.time()-t0:.0f}s) : {pre_dir} (mmap, prêt pour train)")

HTTP Request: GET https://huggingface.co/api/models/openai/whisper-large-v3-turbo/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD http


Pré-calcul (83s) : C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh Anta DIOP de DAKAR\Desktop\B2B\NdapuSpeech\data\processed\ndapuspeech_ds_preprocessed (mmap, prêt pour train)


---
## Étape 5 — Fine-tuning Whisper optimisé 

Config active : `group_by_length` + `dataloader_num_workers=2` + `pin_memory` (16Go RAM),
`attn_implementation=sdpa`, LoRA + 4-bit (8Go VRAM). `torch.compile` reste opt-in.

Entraînement complet (pas de `max_steps`) ; eval plafonnée à `eval_max_samples`
échantillons régulièrement espacés pour garder un signal WER rapide.

In [7]:
cfg = TrainingConfig.from_yaml()
print(f"group_by_length={cfg.group_by_length} workers={cfg.dataloader_num_workers} "
      f"pin_memory={cfg.dataloader_pin_memory} attn={cfg.attn_implementation} "
      f"4bit={cfg.load_in_4bit} compile={cfg.torch_compile}")
print(cfg)

group_by_length=True workers=2 pin_memory=True attn=sdpa 4bit=True compile=False
TrainingConfig(data_dir=WindowsPath('C:/Users/ROG ZEPHYRUS G14/OneDrive - Université Cheikh Anta DIOP de DAKAR/Desktop/B2B/NdapuSpeech/data/processed/ndapuspeech_ds'), output_dir=WindowsPath('C:/Users/ROG ZEPHYRUS G14/OneDrive - Université Cheikh Anta DIOP de DAKAR/Desktop/B2B/NdapuSpeech/models/ndapuspeech-v1'), base_model='openai/whisper-large-v3-turbo', tokenizer_dir=None, extended_tokenizer=False, num_epochs=20, batch_size=4, grad_accum=4, lr_decoder=2e-05, lr_encoder=1e-05, warmup_steps=100, patience=5, noise_dir=None, use_lora=True, resume_from=None, max_steps=None, eval_steps=200, save_steps=400, seed=42, push_to_hub=None, fp16=True, bf16=False, hub_dataset='ndapuspeech/ndapuspeech-dataset', preprocessed_dir=WindowsPath('C:/Users/ROG ZEPHYRUS G14/OneDrive - Université Cheikh Anta DIOP de DAKAR/Desktop/B2B/NdapuSpeech/data/processed/ndapuspeech_ds_preprocessed'), group_by_length=True, dataloader_num_

In [8]:
from pathlib import Path as _Path
t0 = time.time()
_done = _Path(str(cfg.output_dir)) / "training_complete.json"
if _done.exists():
    out_dir = cfg.output_dir
    print(f"[SKIP] Entrainement deja termine : {out_dir} ({(time.time()-t0):.0f}s)")
else:
    out_dir = train_model(cfg)
    print(f"\nModele sauvegarde : {out_dir} ({(time.time()-t0)/60:.0f} min)")


W1017 18:07:34.019000 13948 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
HTTP Request: GET https://huggingface.co/api/models/openai/whisper-large-v3-turbo/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/processor_config.json 

trainable params: 13,107,200 || all params: 821,985,280 || trainable%: 1.5946


Eval capped to 200 evenly-spaced samples for speed
[WIN] dataloader_num_workers=2 forced to 0 (Windows spawn unsafe)
Train samples: 41567, Val samples: 200


Step,Training Loss,Validation Loss,Wer,Cer
200,7.209664,No log,132.199546,89.292487
400,5.367141,No log,71.504157,37.286652
600,4.896117,No log,65.457294,35.667396
800,4.661021,No log,60.317460,31.145150
1000,4.505179,No log,57.369615,28.402626
1200,4.369893,No log,56.991686,27.760759
1400,4.463411,No log,54.950869,28.344274
1600,4.191045,No log,52.380952,24.857768
1800,3.857106,No log,51.700680,25.762217
2000,3.906486,No log,49.281935,23.326039


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more informa

KeyboardInterrupt: 

---
## Étape 6 — Entraînement KenLM n-gram

In [ ]:
from pathlib import Path
arpa_path = Path(config.DATA["kenlm"]["output_arpa"])
bin_path = Path(config.DATA["kenlm"]["output_bin"])
if not (arpa_path.exists() and bin_path.exists()):
    arpa_path, bin_path = train_kenlm()
print(f"ARPA : {arpa_path}")
print(f"BIN  : {bin_path}")

---
## Étape 7 — Évaluation WER / CER vs M-Kiriku

In [ ]:
from ndapuspeech.data import load_jsonl

EVAL_MAX_SAMPLES = 30
test_all = load_jsonl(config.PROCESSED_DATA_DIR / 'ndapuspeech_ds' / 'test.jsonl')
test_samples = test_all[:EVAL_MAX_SAMPLES] if EVAL_MAX_SAMPLES else test_all
print(f'{len(test_samples)} echantillons de test (sur {len(test_all)})')
out_dir = cfg.output_dir
summaries = evaluation.evaluate_and_compare(
    models=[
        str(out_dir),
        'AIHubSN/M-Kiriku-ASR',
    ],
    samples=test_samples,
    batch_size=4,
)


In [ ]:
import pandas as pd

rows = [
    {'modèle': m, 'WER': s['overall']['wer'], 'CER': s['overall']['cer']}
    for m, s in summaries.items()
]

df = pd.DataFrame(rows)
display(df)

baseline_wer = summaries.get('AIHubSN/M-Kiriku-ASR', {}).get('overall', {}).get('wer', None)
ours_wer = summaries.get(str(out_dir), {}).get('overall', {}).get('wer', None)
if baseline_wer and ours_wer:
    verdict = 'BEATS baseline' if ours_wer < baseline_wer else 'sous la baseline'
    print(f"\nVerdict : {verdict}")
    print(f"  M-Kiriku : {baseline_wer:.2f}%")
    print(f"  NdapuSpeech : {ours_wer:.2f}%")

---
## Étape 8 — Inférence sur des fichiers audio

In [ ]:
import glob

seg_dir = config.RAW_DATA_DIR / "kallaama" / "segments"
audio_files = sorted(glob.glob(str(seg_dir / "*.wav")))[:3]
if not audio_files:
    print(f'Aucun fichier audio trouve dans {seg_dir} ')
else:
    results = inference.transcribe_files(
        model_path=str(out_dir),
        audio_paths=audio_files,
        lm_path=None,
    )
    for path, r in results.items():
        print(f'\n--- {path} ---')
        print(r['text'])


---
## Résumé

Le pipeline complet est terminé. Pour un entraînement sérieux :

1. Pré-calcul fait une fois (étape 4b, Arrow `mmap` — ne jamais `flatten()` en RAM)
2. Entraînement complet (base tokenizer, labels free-transcription, eval plafonnée)
3. Lancez 20 époques sur RTX 5060 8Go (SDPA + LoRA 4-bit + `group_by_length`)
4. Poussez le modèle sur HuggingFace Hub

```python
# Push vers Hub (décommenter)
# from huggingface_hub import login; login()
# model.push_to_hub('ndapuspeech/ndapuspeech-asr')
# processor.push_to_hub('ndapuspeech/ndapuspeech-asr-v1')
```